# Huấn luyện & Đánh giá VEC trên Google Colab
### Khung tối ưu hóa có ràng buộc: PID-Lagrangian DQN (Đề xuất) vs DuelingDQN (Baseline [A])

> **Lưu ý thiết lập Runtime:** Chọn **Runtime → Change runtime type → CPU** (Colab 2 vCPU).
> *(Mạng neural MLP kích thước nhỏ; điểm nghẽn 80% là vòng lặp mô phỏng môi trường Python nên GPU không giúp tăng tốc).* 
> Mọi kết quả, model `.pth` và đồ thị sẽ được tự động lưu lên Google Drive để không bị mất khi phiên bị ngắt.

## 1. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Thư mục lưu trữ chính trên Google Drive
DRIVE_DIR = Path('/content/drive/MyDrive/vec_paper1')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = DRIVE_DIR / 'runs'
EVAL_DIR = DRIVE_DIR / 'eval'
PLOTS_DIR = DRIVE_DIR / 'plots'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Đã kết nối Google Drive: {DRIVE_DIR}")

## 2. Đồng bộ Mã nguồn & Dữ liệu Rio Bus
- Clone repository từ GitHub (hoặc `git pull` nếu đã có sẵn).
- Giải nén bộ dữ liệu vết GPS thật Rio Bus `riobus_processed.tgz` (nếu có trên Drive) vào `data/processed/riobus/`.

In [ ]:
%cd /content
!test -d LabEdgeComputing_Paper1 || git clone https://github.com/Linh126-minari/LabEdgeComputing_Paper1.git
%cd /content/LabEdgeComputing_Paper1
!git pull

# Cài đặt các thư viện cần thiết
!pip install -q -r requirements.txt

# Giải nén dữ liệu Rio Bus nếu có sẵn trên Drive
tgz_path = DRIVE_DIR / 'riobus_processed.tgz'
if tgz_path.exists():
    print(f"Giải nén dữ liệu từ {tgz_path}...")
    !tar -xzf {tgz_path}
else:
    print("Lưu ý: Chưa thấy riobus_processed.tgz trên Drive. Nếu chỉ chạy miền Synthetic thì không cần file này.")

!ls -la data/processed/riobus/ 2>/dev/null || true
!echo "Số vCPU hiện tại:"; nproc

## 3. Kiểm tra Tính đúng đắn của Thuật toán (Unit Tests)
Kiểm tra nhanh bộ điều khiển PID-Lagrangian và cơ chế giải nghiệm đóng KKT DVFS (~10 giây).

In [ ]:
!python -m pytest tests/test_pid_lagrangian.py tests/test_energy_dvfs.py -v

## 4. Khởi chạy Huấn luyện Lưới Thực nghiệm
- **Thuật toán:** `lagrangian_dqn` (Đề xuất) và `dueling_dqn` (Baseline [A]). *(Đã loại bỏ MADDPG để tiết kiệm thời gian và tài nguyên).*
- **Miền dữ liệu:** `synthetic` và `real` (Trace Rio Bus).
- **Cập nhật PID:** $\lambda$ được cập nhật mượt sau mỗi episode với damping chống vọt lố.

> Lệnh chạy nền `nohup` giúp bạn có thể tắt tab trình duyệt mà Colab vẫn tiếp tục chạy. Bạn có thể kiểm tra trước kế hoạch bằng `--dry-run`.

In [ ]:
# Xem ước lượng thời gian chạy bằng --dry-run
!python scripts/run_experiments.py --dry-run --agents lagrangian_dqn dueling_dqn --domains synthetic real --seeds 0 1 2 --episodes 150 --jobs 2 --out-root {RUNS_DIR}

In [ ]:
# CHẠY THẬT (Chạy nền, ghi log vào Drive)
!nohup python scripts/run_experiments.py --agents lagrangian_dqn dueling_dqn --domains synthetic real --seeds 0 1 2 --episodes 150 --jobs 2 --out-root {RUNS_DIR} > {DRIVE_DIR}/launcher_train.log 2>&1 &

## 5. Theo dõi Tiến độ Huấn luyện
Chạy ô này bất cứ lúc nào để xem tiến độ và số lượng model đã hoàn thành.

In [ ]:
import time
from IPython.display import clear_output

# Kiểm tra số lượng model đã lưu (Tổng cộng 2 thuật toán x 2 miền x 3 seeds = 12 models)
models = list(RUNS_DIR.glob("*/*/seed*/agent_model.pth"))
print(f"Số lượng models đã hoàn thành: {len(models)} / 12")
for m in models:
    parts = m.parts
    print(f"  ✓ {parts[-4]} | {parts[-3]} | {parts[-2]}")

print("\n--- 20 DÒNG LOG MỚI NHẤT TỪ LAUNCHER ---")
!tail -n 20 {DRIVE_DIR}/launcher_train.log

## 6. Đánh giá Ma trận Chuyển giao Sim-to-Real (Evaluation)
Sau khi các model đã train xong, chạy lệnh đánh giá chung (dùng metric chi phí thật $C_{total}$, không train lại).

In [ ]:
for agent in ['lagrangian_dqn', 'dueling_dqn']:
    print(f"\n================ Đánh giá Sim-to-Real cho {agent} ================")
    !python scripts/sim2real_study.py --runs-dir {RUNS_DIR} --agent {agent} --seeds 0 1 2 --eval-episodes 10 --output_dir {EVAL_DIR}

!ls -la {EVAL_DIR}

## 7. Xuất Toàn bộ Biểu đồ So sánh Chuẩn Bài báo Khoa học
Chạy script vẽ tổng hợp để xuất các đồ thị có độ phân giải cao (300 DPI):
1. `fig1_convergence_comparison.png`: So sánh tốc độ hội tụ chi phí và tỷ lệ vi phạm ràng buộc giữa PID-Lagrangian và DuelingDQN.
2. `fig2_pid_multiplier_dynamics.png`: Động thái của nhân tử PID (chứng minh khả năng dập tắt dao động và chống vọt lố).
3. `fig3_sim2real_transfer.png`: Biểu đồ cột ma trận chuyển giao Sim-to-Real 2x2.

In [ ]:
!python scripts/plot_comparison.py --runs-dir {RUNS_DIR} --eval-dir {EVAL_DIR} --out-dir {PLOTS_DIR} --window 10

## 8. Hiển thị Trực quan các Biểu đồ Trực tiếp trên Notebook

In [ ]:
from IPython.display import Image, display

fig1 = PLOTS_DIR / 'fig1_convergence_comparison.png'
fig2 = PLOTS_DIR / 'fig2_pid_multiplier_dynamics.png'
fig3 = PLOTS_DIR / 'fig3_sim2real_transfer.png'

if fig1.exists():
    print("\n--- HÌNH 1: HỘI TỤ OBJECTIVE & RÀNG BUỘC (SYNTHETIC VS REAL TRACE) ---")
    display(Image(str(fig1), width=950))

if fig2.exists():
    print("\n--- HÌNH 2: ĐỘNG THÁI NHÂN TỬ ĐỐI NGŨ PID-LAGRANGIAN (CHỐNG DAO ĐỘNG) ---")
    display(Image(str(fig2), width=900))

if fig3.exists():
    print("\n--- HÌNH 3: MA TRẬN CHUYỂN GIAO SIM-TO-REAL GIỮA 2 THUẬT TOÁN ---")
    display(Image(str(fig3), width=900))

## 9. Bảng Tổng hợp Kết quả cho Bài báo (LaTeX / Markdown Table)

In [ ]:
import pandas as pd

csv_lagrangian = EVAL_DIR / 'transfer_matrix_lagrangian_dqn.csv'
csv_dueling = EVAL_DIR / 'transfer_matrix_dueling_dqn.csv'

if csv_lagrangian.exists() and csv_dueling.exists():
    df_lag = pd.read_csv(csv_lagrangian).assign(Agent='PID-LagrangianDQN (Đề xuất)')
    df_due = pd.read_csv(csv_dueling).assign(Agent='DuelingDQN (Baseline [A])')
    summary = pd.concat([df_lag, df_due], ignore_index=True)
    
    # Định dạng các cột chính
    cols = ['Agent', 'train', 'test', 'objective_cents_mean', 'viol_T_mean', 'viol_E_mean', 'offload_ratio_mean', 'aux_share_mean', 'block_length_aux_mean']
    display_df = summary[cols].copy()
    display_df['viol_T_mean'] = (display_df['viol_T_mean'] * 100).round(2).astype(str) + '%'
    display_df['objective_cents_mean'] = display_df['objective_cents_mean'].round(4)
    
    print("\nBẢNG SỐ LIỆU ĐỐI SÁNH SIM-TO-REAL CHUẨN CÔNG BỐ:")
    display(display_df)
else:
    print("Chưa đủ dữ liệu CSV để hiển thị bảng. Hãy chạy Bước 6 trước.")